# Case-study NB1 — Peeling-chain-like setup and NEST smoke test

This notebook creates a deterministic, fixed-seed (`42`) Elliptic case-study graph, validates the 64↔8 nested-prototype mechanics, and performs only a short NEST smoke test. It never modifies its read-only Kaggle inputs. Its output is a frozen graph and manifest for a later training notebook.

> Methodological boundary: this is a **peeling-chain-like** structural illustration, not a claim that the Elliptic labels establish a confirmed laundering typology. Tier 8 is a nested-prototype budget in the inherited implementation, not a physically smaller Client-B encoder.

In [ ]:
# 1. Preflight, discovery, and provenance
from pathlib import Path
import os, glob, hashlib, json, time, random, warnings, copy
from datetime import datetime, timezone

DATA_FILES = ('elliptic_txs_features.csv', 'elliptic_txs_edgelist.csv', 'elliptic_txs_classes.csv')
# Accept the clean NEST-main release names and the older working-copy names.
SOURCE_ALIASES = {'notebook1.ipynb': ('nest_data_models_baselines.ipynb', 'notebook1.ipynb'), 'notebook2.ipynb': ('nest_rank_aware_aggregation.ipynb', 'notebook2.ipynb')}
SEED = 42
OUT = Path('/kaggle/working/peeling_chain_case_study')
OUT.mkdir(parents=True, exist_ok=True)

def find_unique(filename):
    hits = sorted(Path(p) for p in glob.glob('/kaggle/input/**/' + filename, recursive=True))
    if not hits:
        raise FileNotFoundError(f'Missing required input: {filename}')
    if len(hits) > 1:
        print(f'[warning] {filename}: using {hits[0]} among {len(hits)} matches')
    return hits[0]

def find_source(aliases, logical_name):
    for filename in aliases:
        hits = sorted(Path(p) for p in glob.glob('/kaggle/input/**/' + filename, recursive=True))
        if hits:
            print(f'[source] {logical_name} resolved to {hits[0].name}')
            return hits[0]
    raise FileNotFoundError(f'Missing NEST source for {logical_name}; accepted names: {aliases}')

PATHS = {name: find_unique(name) for name in DATA_FILES}
PATHS.update({logical: find_source(aliases, logical) for logical, aliases in SOURCE_ALIASES.items()})
def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

print('Resolved inputs:')
for name, path in PATHS.items():
    print(f'  {name}: {path}')
print('Source hashes:')
SOURCE_HASHES = {name: sha256(PATHS[name]) for name in ('notebook1.ipynb', 'notebook2.ipynb')}
for name, digest in SOURCE_HASHES.items(): print(f'  {name}: {digest}')

try:
    import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
    import torch_geometric
    from torch_geometric.data import Data
    from sklearn.model_selection import train_test_split
    import scipy, matplotlib, nbformat
except Exception as exc:
    raise RuntimeError('Required package import failed. Enable a Kaggle image with PyTorch Geometric before continuing.') from exc

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}; torch={torch.__version__}; pyg={torch_geometric.__version__}')
START = time.time()


In [ ]:
# 2. Import only required inherited definitions (never execute source notebooks wholesale)
# The selected source cells are executed verbatim, preserving the original implementations.
def source_of_marker(nb_path, marker):
    nb = nbformat.read(nb_path, as_version=4)
    matches = [''.join(cell.source) for cell in nb.cells if cell.cell_type == 'code' and marker in ''.join(cell.source)]
    if len(matches) != 1:
        raise RuntimeError(f'Expected exactly one source cell for {marker!r}; found {len(matches)}')
    return matches[0]

# Notebook 2 has the current common definitions and the NEST runner. Notebook 1 supplies fixed-tier FedProto.
N2 = PATHS['notebook2.ipynb']; N1 = PATHS['notebook1.ipynb']
MARKERS_N2 = [
    'from dataclasses import dataclass', 'class SAGEGATEncoder',
    'def evaluate_tuned', 'def ssl_pretrain_client', 'def run_pgfcl',
    'DEVICE_TIER_DIMS = (8, 16, 32, 64)', 'def _all_masks', 'def rank_aware_contrib_agg',
    'FLOAT_BYTES = 4', 'def supervised_round_client_ep'
]
for marker in MARKERS_N2:
    exec(compile(source_of_marker(N2, marker), f'<notebook2:{marker}>', 'exec'), globals())
# These two helpers are copied verbatim from the source split-utility cell.
# That cell also launches an original full-dataset split, so it must not be executed as a whole.
def get_local_edge_index(data: Data, mask: torch.Tensor, device: torch.device) -> torch.Tensor:
    ei = data.edge_index.to(device); m = mask.to(device); keep = m[ei[0]] & m[ei[1]]
    return ei[:, keep]

def get_inductive_edge_index(data, train_mask, test_mask, device):
    ei = data.edge_index.to(device); tr = train_mask.to(device); te = test_mask.to(device); all_m = tr | te
    keep = (te[ei[1]] & all_m[ei[0]]) | (te[ei[0]] & te[ei[1]])
    return ei[:, keep]
exec(compile(source_of_marker(N1, 'def _fixed_tier_effective_cfg'), '<notebook1:fixed-tier>', 'exec'), globals())
print('Inherited definitions loaded without executing experiment cells.')

# Faithful 330-feature Elliptic loader. Unlike the source loader, this has no hard-coded Kaggle path or side effects.
def load_case_elliptic():
    features = pd.read_csv(PATHS['elliptic_txs_features.csv'], header=None)
    edges = pd.read_csv(PATHS['elliptic_txs_edgelist.csv'])
    classes = pd.read_csv(PATHS['elliptic_txs_classes.csv'])
    node_ids = features.iloc[:, 0].to_numpy()
    timesteps = features.iloc[:, 1].to_numpy()
    raw = np.nan_to_num(features.iloc[:, 2:].to_numpy(dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0)
    dev = np.zeros_like(raw)
    for t in np.unique(timesteps):
        mask = timesteps == t; dev[mask] = (raw[mask] - raw[mask].mean(0)) / (raw[mask].std(0) + 1e-8)
    from sklearn.preprocessing import StandardScaler
    x = np.nan_to_num(StandardScaler().fit_transform(np.concatenate([raw, dev], axis=1)).astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
    label_map = dict(zip(classes['txId'], classes['class'].map({'1': 1, '2': 0, 'unknown': -1})))
    id2idx = {nid: i for i, nid in enumerate(node_ids)}
    valid = [(id2idx[u], id2idx[v]) for u, v in zip(edges.iloc[:, 0], edges.iloc[:, 1]) if u in id2idx and v in id2idx]
    if not valid: raise RuntimeError('No valid Elliptic edges were found.')
    edge_index = torch.tensor(valid, dtype=torch.long).t().contiguous()
    y = torch.tensor([label_map.get(nid, -1) for nid in node_ids], dtype=torch.long)
    return Data(x=torch.tensor(x), edge_index=edge_index, y=y, timestep=torch.tensor(timesteps, dtype=torch.long), node_id=torch.tensor(node_ids, dtype=torch.long))

elliptic = load_case_elliptic()
print(f'Elliptic loaded: nodes={elliptic.num_nodes:,}, edges={elliptic.num_edges:,}, features={elliptic.num_node_features}')


In [ ]:
# 3. Deterministic structural screening and role-aware A/B partition
MAX_CANDIDATES, DOWNSTREAM_HOPS, UPSTREAM_HOPS = 100, 4, 1
# With a 30% stratified test split, 14 illicit labels are required to guarantee at least four held-out illicit nodes.
MIN_ILLICIT, MIN_LICIT, MIN_TEST_ILLICIT, MIN_TEST_LICIT = 14, 34, 4, 10
TEST_RATIO = 0.30

src, dst = elliptic.edge_index.cpu().numpy()
n = elliptic.num_nodes
out_adj, in_adj = [[] for _ in range(n)], [[] for _ in range(n)]
for u, v in zip(src, dst): out_adj[int(u)].append(int(v)); in_adj[int(v)].append(int(u))
out_deg = np.fromiter((len(a) for a in out_adj), dtype=int, count=n)
in_deg = np.fromiter((len(a) for a in in_adj), dtype=int, count=n)
degree = out_deg + in_deg

def bfs(seed, adjacency, hops):
    depth, frontier = {int(seed): 0}, [int(seed)]
    for d in range(1, hops + 1):
        nxt = []
        for u in frontier:
            for v in adjacency[u]:
                if v not in depth: depth[v] = d; nxt.append(v)
        frontier = nxt
        if not frontier: break
    return depth

def class_counts(nodes):
    yy = elliptic.y.cpu().numpy()[list(nodes)] if nodes else np.array([], dtype=int)
    return int((yy == 1).sum()), int((yy == 0).sum())

rows, candidates = [], []
illicit = np.where(elliptic.y.cpu().numpy() == 1)[0]
for seed in sorted(illicit, key=lambda i: (-degree[i], int(i)))[:MAX_CANDIDATES]:
    downstream = bfs(seed, out_adj, DOWNSTREAM_HOPS); upstream = bfs(seed, in_adj, UPSTREAM_HOPS)
    nodes = set(downstream) | set(upstream)
    terminals = {u for u, d in downstream.items() if d >= 2 and out_deg[u] <= 1}
    a_nodes = {u for u, d in downstream.items() if d <= 1} | set(upstream)
    b_nodes = terminals
    # Retain only disjoint structural roles; no performance-based selection.
    b_nodes -= a_nodes
    ai, al = class_counts(a_nodes); bi, bl = class_counts(b_nodes); ni, nl = class_counts(nodes)
    path_count = sum(1 for t in terminals if downstream[t] >= 2)
    eligible = ai >= MIN_ILLICIT and al >= MIN_LICIT and bi >= MIN_ILLICIT and bl >= MIN_LICIT and path_count > 0
    score = (int(eligible), path_count, len(terminals), degree[seed], len(nodes))
    induced_edges = int(np.count_nonzero(np.isin(src, list(nodes)) & np.isin(dst, list(nodes))))
    row = dict(seed_idx=int(seed), seed_node_id=int(elliptic.node_id[seed]), nodes=len(nodes), edges=induced_edges, hub_in=int(in_deg[seed]), hub_out=int(out_deg[seed]), hub_degree=int(degree[seed]), max_depth=max(downstream.values()), terminals=len(terminals), hub_intermediate_terminal_paths=path_count, illicit=ni, licit=nl, A_illicit=ai, A_licit=al, B_illicit=bi, B_licit=bl, eligible=bool(eligible), structural_score=repr(score))
    rows.append(row); candidates.append((score, seed, nodes, a_nodes, b_nodes, downstream, row))

screen = pd.DataFrame(rows).sort_values(['eligible', 'hub_intermediate_terminal_paths', 'terminals', 'hub_degree'], ascending=False)
screen.to_csv(OUT / 'candidate_screening.csv', index=False)
if not candidates or not any(c[-1]['eligible'] for c in candidates):
    display(screen.head(20))
    raise RuntimeError('No structurally and label-eligible peeling-chain-like candidate found. Thresholds were not relaxed; inspect candidate_screening.csv.')
best = max((c for c in candidates if c[-1]['eligible']), key=lambda c: c[0])
_, seed_idx, selected_nodes, A_nodes, B_nodes, depths, selection_row = best
selected_nodes = sorted(selected_nodes)
old_to_new = {old: new for new, old in enumerate(selected_nodes)}
keep = np.array([(u in old_to_new and v in old_to_new) for u, v in zip(src, dst)])
case_edges = torch.tensor([[old_to_new[int(u)] for u, v in zip(src[keep], dst[keep])], [old_to_new[int(v)] for u, v in zip(src[keep], dst[keep])]], dtype=torch.long)
case = Data(x=elliptic.x[selected_nodes].clone(), edge_index=case_edges, y=elliptic.y[selected_nodes].clone(), timestep=elliptic.timestep[selected_nodes].clone(), node_id=elliptic.node_id[selected_nodes].clone())
A_case = {old_to_new[u] for u in A_nodes if u in old_to_new}; B_case = {old_to_new[u] for u in B_nodes if u in old_to_new}
print('Selected structural candidate:', selection_row)
print(f'Frozen case graph: nodes={case.num_nodes}, edges={case.num_edges}, A={len(A_case)}, B={len(B_case)}')


In [ ]:
# 4. Fixed split, manifest, tier assertions, and short end-to-end NEST smoke test
def make_client(cid, role_nodes):
    role_nodes = sorted(role_nodes); labels = case.y.cpu().numpy()
    labeled = np.array([u for u in role_nodes if labels[u] >= 0], dtype=int)
    unlabeled = np.array([u for u in role_nodes if labels[u] < 0], dtype=int)
    y = labels[labeled]
    if len(np.unique(y)) != 2:
        raise RuntimeError(f'Client {cid} has no two-class labeled split.')
    tr, te = train_test_split(labeled, test_size=TEST_RATIO, random_state=SEED, stratify=y)
    train_mask, test_mask = torch.zeros(case.num_nodes, dtype=torch.bool), torch.zeros(case.num_nodes, dtype=torch.bool)
    train_mask[torch.tensor(np.concatenate([tr, unlabeled]), dtype=torch.long)] = True
    test_mask[torch.tensor(te, dtype=torch.long)] = True
    ti, tl = int((labels[tr] == 1).sum()), int((labels[tr] == 0).sum())
    vi, vl = int((labels[te] == 1).sum()), int((labels[te] == 0).sum())
    if ti < 2 or tl < 2 or vi < MIN_TEST_ILLICIT or vl < MIN_TEST_LICIT:
        raise RuntimeError(f'Client {cid} fails post-split class requirements: train illicit/lic it={ti}/{tl}; test illicit/lic it={vi}/{vl}')
    return {'id': cid, 'train_mask': train_mask, 'test_mask': test_mask, 'n_train': int(train_mask.sum()), 'n_test': int(test_mask.sum()), 'role_nodes': role_nodes, 'counts': {'train_illicit':ti, 'train_licit':tl, 'test_illicit':vi, 'test_licit':vl}}

clients = [make_client(0, A_case), make_client(1, B_case)]
for name, client in zip(('A', 'B'), clients): print(name, client['counts'], 'n_train=', client['n_train'], 'n_test=', client['n_test'])
tier_dims = (8, 64)
tier_map = assign_device_tiers(clients, tier_dims=tier_dims, seed=SEED)
assert tier_map == {0: 64, 1: 8}, f'Expected Client A=64 and Client B=8; got {tier_map}'
p = {0: torch.arange(64, dtype=torch.float), 1: torch.arange(64, dtype=torch.float)}
assert torch.equal(truncate_proto(p, 8)[1], p[1][:8])
print('Verified tier map:', tier_map)
print('Verified slicing: B uses z[:, :8] against prototype[:8]; A includes the 8-D and 64-D nested budgets. Shared prefix does not mean equal A/B embeddings.')

manifest = {'created_utc': datetime.now(timezone.utc).isoformat(), 'seed': SEED, 'source_hashes': SOURCE_HASHES, 'selection': selection_row, 'thresholds': {'min_illicit':MIN_ILLICIT, 'min_licit':MIN_LICIT, 'min_test_illicit':MIN_TEST_ILLICIT, 'min_test_licit':MIN_TEST_LICIT, 'test_ratio':TEST_RATIO, 'downstream_hops':DOWNSTREAM_HOPS, 'upstream_hops':UPSTREAM_HOPS}, 'tier_dims': tier_dims, 'tier_map': tier_map, 'case_node_ids': [int(x) for x in case.node_id.tolist()], 'clients': [{k:v for k,v in c.items() if k not in ('train_mask','test_mask')} | {'train_mask_indices': torch.where(c['train_mask'])[0].tolist(), 'test_mask_indices': torch.where(c['test_mask'])[0].tolist()} for c in clients]}
with open(OUT / 'manifest.json', 'w') as f: json.dump(manifest, f, indent=2)
torch.save({'data': case, 'clients': clients, 'manifest': manifest}, OUT / 'frozen_case_graph.pt')

SMOKE_CFG = ExperimentConfig(global_rounds=5, ssl_pretrain_rounds=1, ssl_epochs=20, sup_epochs=25, head_finetune_rounds=1, use_saliency=False, use_calibration=False, seeds=(SEED,), n_clients=2)
smoke_start = time.time()
try:
    smoke = run_ep_fedproto(case, clients, DEVICE, SMOKE_CFG, seed=SEED, tier_dims=tier_dims, verbose=True, label='Case-study NEST smoke')
except Exception as exc:
    (OUT / 'smoke_log.txt').write_text(f'SMOKE FAILED: {type(exc).__name__}: {exc}\n', encoding='utf-8')
    raise
metrics = smoke[0]
numeric = [metrics.get(k, float('nan')) for k in ('f1','auc','prec','rec','acc')]
if not np.isfinite(numeric).all(): raise RuntimeError(f'Smoke returned non-finite metrics: {metrics}')
smoke_seconds = time.time() - smoke_start
results = {'status':'passed', 'smoke_wall_time_s':smoke_seconds, 'total_notebook_wall_time_s':time.time()-START, 'metrics':{k: (v.tolist() if hasattr(v, 'tolist') else v) for k,v in metrics.items() if k not in ('cm','ece_bins')}, 'tier_map':tier_map, 'estimated_full_nest_seconds': smoke_seconds * (100 / 5)}
with open(OUT / 'smoke_results.json', 'w') as f: json.dump(results, f, indent=2)
(OUT / 'smoke_log.txt').write_text('SMOKE PASSED\n' + json.dumps(results, indent=2), encoding='utf-8')
print('\n=== NB2 HANDOFF ===')
print(f'Case graph: {case.num_nodes} nodes, {case.num_edges} edges')
print('A/B counts:', clients[0]['counts'], clients[1]['counts'])
print('Tier map:', tier_map, '| smoke seconds:', round(smoke_seconds, 1))
print('Artifacts:', OUT / 'frozen_case_graph.pt', OUT / 'manifest.json', OUT / 'smoke_results.json')
print('Ready for fixed-seed three-arm training only if all validations above passed.')
